In [6]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(r"C:\Users\16960\Desktop\期末论文\模型搭建\LLM\data")
MANIFEST_PATH = DATA_DIR / "llm_dataset_manifest.json"
SPLIT_NAME = "train"      # 可改为 train / val / test
MAX_ROWS = 1000           # 先少量观察，避免一次性读取超大 JSONL
OUT_CSV = Path(r"C:\Users\16960\Desktop\期末论文\模型搭建\LLM\数据观测.csv")


def load_jsonl(filepath, max_rows=None):
    """读取 JSONL 文件，返回记录列表。"""
    records = []
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
            if max_rows is not None and len(records) >= max_rows:
                break
    return records


def get_nested(obj, path, default=np.nan):
    cur = obj
    for key in path.split("."):
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def extract_features(record):
    """从新版 strict JSONL 中提取结构化特征。"""
    features = record.get("features", {})
    return {
        "数据切分": SPLIT_NAME,
        "日期": record.get("date"),
        "样本ID": record.get("sample_id"),
        "标签": record.get("label"),
        "标签文本": record.get("output"),
        "schema_version": record.get("schema_version"),
        "prompt_version": record.get("prompt_version"),
        "observation_policy": record.get("observation_policy"),
        "链位置": record.get("chain_position"),
        "链有效长度": record.get("chain_valid_length"),

        "出发机场": get_nested(features, "current.origin"),
        "到达机场": get_nested(features, "current.dest"),
        "承运人": get_nested(features, "current.carrier"),
        "航班号": get_nested(features, "current.flight_number"),
        "飞机尾号": get_nested(features, "current.tail"),
        "计划起飞时间": get_nested(features, "current.dep_time"),
        "计划到达时间": get_nested(features, "current.arr_time"),
        "计划飞行分钟": get_nested(features, "current.elapsed_min"),

        "出发地气温": get_nested(features, "current.origin_weather.temp"),
        "出发地降水": get_nested(features, "current.origin_weather.prcp"),
        "出发地风速": get_nested(features, "current.origin_weather.wspd"),
        "到达地气温": get_nested(features, "current.dest_weather.temp"),
        "到达地降水": get_nested(features, "current.dest_weather.prcp"),
        "到达地风速": get_nested(features, "current.dest_weather.wspd"),

        "前序出发延误分钟": get_nested(features, "tail_chain.prev_dep_delay"),
        "前序到达延误分钟": get_nested(features, "tail_chain.prev_arr_delay"),
        "计划过站缓冲分钟": get_nested(features, "tail_chain.turnaround_slack_min"),
        "剩余缓冲分钟": get_nested(features, "tail_chain.remaining_slack_min"),
        "传播压力": get_nested(features, "tail_chain.propagation_pressure"),
        "出发延误趋势": get_nested(features, "tail_chain.dep_delay_trend"),

        "同航段近5班数量": get_nested(features, "route_context.count"),
        "同航段近5班延误率": get_nested(features, "route_context.delay_rate"),
        "同航段近5班平均延误": get_nested(features, "route_context.mean_delay"),
        "同航段近5班最大延误": get_nested(features, "route_context.max_delay"),

        "反向航段数量": get_nested(features, "reverse_route_context.count"),
        "反向航段延误率": get_nested(features, "reverse_route_context.delay_rate"),

        "出发机场60分钟数量": get_nested(features, "airport_context.origin_60m.count"),
        "出发机场60分钟延误率": get_nested(features, "airport_context.origin_60m.delay_rate"),
        "出发机场60分钟平均延误": get_nested(features, "airport_context.origin_60m.mean_delay"),
        "到达机场60分钟数量": get_nested(features, "airport_context.dest_60m.count"),
        "到达机场60分钟延误率": get_nested(features, "airport_context.dest_60m.delay_rate"),
        "同承运人同出发机场60分钟数量": get_nested(features, "airport_context.carrier_origin_60m.count"),
        "同承运人同出发机场60分钟延误率": get_nested(features, "airport_context.carrier_origin_60m.delay_rate"),

        "200km邻近出发机场数量": get_nested(features, "nearby_airport_context.near_origin_60m.count"),
        "200km邻近出发机场延误率": get_nested(features, "nearby_airport_context.near_origin_60m.delay_rate"),
        "200km邻近到达机场数量": get_nested(features, "nearby_airport_context.near_dest_60m.count"),
        "200km邻近到达机场延误率": get_nested(features, "nearby_airport_context.near_dest_60m.delay_rate"),

        "链式风险": get_nested(features, "propagation_risk.chain_pressure"),
        "机场风险": get_nested(features, "propagation_risk.airport_pressure"),
        "航段风险": get_nested(features, "propagation_risk.route_pressure"),
        "综合风险": get_nested(features, "propagation_risk.composite_risk"),

        "compact_text": record.get("compact_text", ""),
    }


with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

jsonl_path = DATA_DIR / f"{SPLIT_NAME}.jsonl"
records = load_jsonl(jsonl_path, max_rows=MAX_ROWS)
df = pd.DataFrame([extract_features(x) for x in records])

print("当前数据版本:", manifest.get("schema_version"), manifest.get("prompt_version"))
print("观测策略:", manifest.get("observation_policy"))
print("读取文件:", jsonl_path)
print("读取样本数:", len(df))
print("全量行数:", manifest["splits"][SPLIT_NAME]["rows"])
df.to_csv("样本汉化.csv", index=False, encoding="utf-8-sig")
display(df.head())

当前数据版本: chain_llm_strict propagation_capsule_strict_v1
观测策略: strict_actual_event_before_prediction
读取文件: C:\Users\16960\Desktop\期末论文\模型搭建\LLM\data\train.jsonl
读取样本数: 1000
全量行数: 4022382


,数据切分,日期,样本ID,标签,标签文本,schema_version,prompt_version,observation_policy,链位置,链有效长度,...,同承运人同出发机场60分钟延误率,200km邻近出发机场数量,200km邻近出发机场延误率,200km邻近到达机场数量,200km邻近到达机场延误率,链式风险,机场风险,航段风险,综合风险,compact_text
0,train,2024-01-01,240101_000000_1,0,正常,chain_llm_strict,propagation_capsule_strict_v1,strict_actual_event_before_prediction,1,4,...,0.0,28,0.071429,22,0.090909,0.00,0.000000,0.0,0.000000,当前: AVL->FLL; 12:30起飞; 14:30到达; G4228; 尾号190NV...
1,train,2024-01-01,240101_000000_2,0,正常,chain_llm_strict,propagation_capsule_strict_v1,strict_actual_event_before_prediction,2,4,...,1.0,32,0.093750,22,0.000000,0.00,0.285714,0.0,0.285714,当前: FLL->AVL; 15:20起飞; 17:21到达; G4259; 尾号190NV...
2,train,2024-01-01,240101_000000_3,1,延误,chain_llm_strict,propagation_capsule_strict_v1,strict_actual_event_before_prediction,3,4,...,NaN,24,0.000000,45,0.022222,0.00,0.000000,0.0,0.000000,当前: AVL->PIE; 18:11起飞; 19:53到达; G41852; 尾号190N...
3,train,2024-01-01,240101_000000_4,1,延误,chain_llm_strict,propagation_capsule_strict_v1,strict_actual_event_before_prediction,4,4,...,0.0,32,0.218750,20,0.100000,0.52,0.000000,0.0,0.520000,当前: PIE->AVL; 20:43起飞; 22:20到达; G4241; 尾号190NV...
4,train,2024-01-01,240101_000001_1,0,正常,chain_llm_strict,propagation_capsule_strict_v1,strict_actual_event_before_prediction,1,4,...,NaN,55,0.054545,1,0.000000,0.00,0.000000,0.0,0.000000,当前: PIE->SGF; 10:00起飞; 11:38到达; G42694; 尾号191N...


In [7]:
missing_df = pd.DataFrame({
    "字段名": df.columns,
    "缺失数量": df.isna().sum().values,
    "缺失比例": (df.isna().mean().values).round(4),
}).sort_values("缺失比例", ascending=False)

display(missing_df)

,字段名,缺失数量,缺失比例
29,出发延误趋势,624,0.624
31,同航段近5班延误率,439,0.439
32,同航段近5班平均延误,439,0.439
33,同航段近5班最大延误,439,0.439
42,同承运人同出发机场60分钟延误率,337,0.337
28,传播压力,313,0.313
25,前序到达延误分钟,313,0.313
27,剩余缓冲分钟,313,0.313
24,前序出发延误分钟,312,0.312
26,计划过站缓冲分钟,312,0.312
